# EDA — Donate user pool

Profile every user who is **ever in the Donate pool** over the 6-month window (TFU users included).

- **Donate pool** = lifetime `is_gifter == 1` (broad / overlapping — includes TFU)
- Intensity grade = quartile of lifetime `avg_gift_usd` within the pool.

Sections:
1. Overview — pool size, monthly trend, intensity distribution
2. Segmentation — categorical breakdowns × intensity quartile
3. Behavioral profile — numeric features by intensity quartile
4. Correlation — raw + leakage-aware clean + outlier-trimmed
5. Stickiness — months_observed, repeat rate, retention curve


## 0. Setup

In [ ]:
# --- Colab auth + BigQuery client ---
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')

PROJECT_ID      = 'nf-bifrost'
DATASET         = 'nf-muses.muses'
MONTHLY_TABLE   = f'{DATASET}.tfu_user_monthly'
LIFETIME_TABLE  = f'{DATASET}.tfu_user_lifetime'

client = bigquery.Client(project=PROJECT_ID)


In [ ]:
# Load monthly + lifetime tables
df = client.query(f'SELECT * FROM `{MONTHLY_TABLE}`').to_dataframe()
df['data_month'] = pd.to_datetime(df['data_month'])

# Pool flags at the monthly grain
df['has_gift'] = ((df['total_tip_count'].fillna(0)
                   + df['total_box_count'].fillna(0)
                   + df['total_wheel_count'].fillna(0)) > 0).astype(int)
df['has_follow_bet'] = (df['total_follow_bet_count'].fillna(0) > 0).astype(int)
df['total_gift_usd'] = (df['total_tip_usd'].fillna(0)
                        + df['total_box_usd'].fillna(0)
                        + df['total_wheel_usd'].fillna(0))

users = client.query(f'SELECT * FROM `{LIFETIME_TABLE}`').to_dataframe()
users['has_gift']       = users['is_gifter']
users['has_follow_bet'] = users['is_follow_bet']
users['avg_gift_usd']   = (users['avg_tip_usd'].fillna(0)
                           + users['avg_box_usd'].fillna(0)
                           + users['avg_wheel_usd'].fillna(0))

print('monthly rows :', len(df), ' months :', sorted(df.data_month.dt.strftime('%Y-%m').unique()))
print('lifetime users:', len(users))


## 1. Overview — Donate pool

In [ ]:
# Restrict to Donate pool (gifter — TFU included)
pool_u = users[users['has_gift'] == 1].copy()
pool_m = df[df['has_gift'] == 1].copy()

# Intensity quartiles on lifetime intensity (avg_gift_usd)
pool_u['intensity'] = pd.qcut(pool_u['avg_gift_usd'].rank(method='first'),
                              q=4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])
intensity_order = ['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']

print('Donate pool — unique users :', len(pool_u))
print('Donate pool — monthly rows :', len(pool_m))
print('\nIntensity quartile cut-points (avg_gift_usd):')
print(pool_u.groupby('intensity')['avg_gift_usd']
            .agg(['count', 'min', 'median', 'max']).round(2))


In [ ]:
# Monthly pool size + share of all users
overview = (df.groupby('data_month')
              .agg(all_users  = ('cust_id',  'nunique'),
                   pool_users = ('has_gift', 'sum'))
              .reset_index())
overview['pool_share'] = overview['pool_users'] / overview['all_users']
print(overview)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=overview, x='data_month', y='pool_users', ax=axes[0], color='#4c78a8')
axes[0].set_title('Donate pool size per month')
axes[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=overview, x='data_month', y='pool_share',
             marker='o', linewidth=2, ax=axes[1])
axes[1].set_title('Donate share of all active users')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()


In [ ]:
# Intensity distribution — lifetime
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(np.log1p(pool_u['avg_gift_usd']), bins=60, ax=axes[0], color='#4c78a8')
axes[0].set_title('log1p(avg_gift_usd) — Donate pool')
axes[0].set_xlabel('log1p(avg_gift_usd)')

sns.boxplot(data=pool_u, x='intensity', y='avg_gift_usd',
            order=intensity_order, ax=axes[1], showfliers=False)
axes[1].set_yscale('symlog')
axes[1].set_title('avg gift USD per observed month by quartile')
plt.tight_layout(); plt.show()


## 2. Segmentation — categorical mix of the Donate pool

For each categorical column, one row per segment value (counts are **unique users in the pool**):

| column | description |
|---|---|
| `user_count` | users in this segment of the pool |
| `pool_share` | segment's share of pool users |
| `high_count` | users that are in the top intensity quartile (Q4) |
| `high_rate`  | `high_count / user_count` — how dense Q4 is in the segment |
| `high_share` | segment's share of all Q4 users |


In [ ]:
categorical_cols = [
    'account_age_tier', 'watch_bucket', 'sessions_bucket',
    'time_segment', 'day_segment',
    'stream_type_pref', 'device_pref',
    'league_segment',
]


In [ ]:
def segment_table(pool_df, col):
    total_users = len(pool_df)
    total_high  = (pool_df['intensity'] == 'Q4 (high)').sum()
    t = (pool_df.assign(is_high=(pool_df['intensity'] == 'Q4 (high)').astype(int))
                .groupby(col)
                .agg(user_count=('cust_id',  'nunique'),
                     high_count=('is_high',  'sum'))
                .reset_index())
    t['pool_share'] = t['user_count'] / total_users
    t['high_rate']  = t['high_count'] / t['user_count']
    t['high_share'] = t['high_count'] / total_high
    t = t.sort_values('high_share', ascending=False).reset_index(drop=True)
    return t

for col in categorical_cols:
    print(f'\n--- {col} ---')
    tab = segment_table(pool_u, col)
    fmt = tab.copy()
    fmt['user_count'] = fmt['user_count'].map('{:,}'.format)
    fmt['high_count'] = fmt['high_count'].map('{:,}'.format)
    fmt['pool_share'] = fmt['pool_share'].map('{:.2%}'.format)
    fmt['high_rate']  = fmt['high_rate'].map('{:.2%}'.format)
    fmt['high_share'] = fmt['high_share'].map('{:.2%}'.format)
    print(fmt.to_string(index=False))


In [ ]:
def segment_plot(pool_df, col, pool_name):
    t = segment_table(pool_df, col)
    fig, ax1 = plt.subplots(figsize=(9, 4))
    sns.barplot(data=t, x=col, y='high_share', color='#9ecae1', ax=ax1)
    ax1.set_ylabel('high_share (bars)', color='#3182bd')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax1.tick_params(axis='x', rotation=30)
    for label in ax1.get_xticklabels():
        label.set_ha('right')

    ax2 = ax1.twinx()
    ax2.plot(range(len(t)), t['high_rate'].values, color='#e6550d', marker='o', linewidth=2)
    ax2.set_ylabel('high_rate (line)', color='#e6550d')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
    plt.title(f'{pool_name} — {col}  (unique users)')
    plt.tight_layout(); plt.show()


In [ ]:
for col in categorical_cols:
    segment_plot(pool_u, col, 'Donate pool')


## 3. Behavioral profile — by intensity quartile

Compare quartiles inside the Donate pool on numeric behavior.

In [ ]:
numeric_features = [
    # Watch
    'avg_watch_sec', 'avg_watch_sec_per_session',
    # Chat
    'avg_messages', 'avg_chat_sessions', 'avg_bullet_sec', 'avg_chatroom_sec',
    # Gifting
    'avg_tip_count', 'avg_box_count', 'avg_wheel_count',
    'avg_tip_usd', 'avg_box_usd', 'avg_wheel_usd', 'avg_gift_usd',
    # Betting
    'avg_bet_count', 'avg_member_to', 'avg_bdw_bet_count', 'avg_follow_bet_count',
    # Breadth / loyalty
    'breadth_score', 'avg_sessions_count', 'distinct_streamers', 'months_observed',
]


In [ ]:
profile = (pool_u.groupby('intensity')[numeric_features]
                 .agg(['median', 'mean']).round(2))
profile.loc[intensity_order]


In [ ]:
# Boxplots (log scale) per numeric feature by intensity quartile
plot_df = pool_u.copy()

n = len(numeric_features); ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    ax = axes[i]
    data = plot_df[[feat, 'intensity']].copy()
    data[feat] = data[feat].clip(lower=0) + 1
    sns.boxplot(data=data, x='intensity', y=feat, order=intensity_order,
                ax=ax, showfliers=False)
    ax.set_yscale('log')
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Donate pool — behavioral profile by intensity quartile', y=1.02)
plt.tight_layout(); plt.show()


## 4. Correlation — numeric features vs `avg_gift_usd`

Within the Donate pool: Spearman correlation between behavior features and the
continuous intensity column `avg_gift_usd`.

In [ ]:
def corr_heatmap(frame, pool_name, target='avg_gift_usd'):
    cols = [c for c in numeric_features if c != target] + [target]
    corr = frame[cols].corr(method='spearman')
    plt.figure(figsize=(11, 9))
    sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, annot=False)
    plt.title(f'Spearman correlation — {pool_name} (n={len(frame):,})')
    plt.tight_layout(); plt.show()

    signed = corr[target].drop(target)
    top = signed.reindex(signed.abs().sort_values(ascending=False).index).head(15)
    print(f'\nTop corr with {target} — {pool_name} (signed, sorted by |corr|):')
    print(top.round(3))

corr_heatmap(pool_u, 'Donate pool')


### 4b. Leakage-aware (clean) correlation

The intensity target `avg_gift_usd` is built from gifting/betting columns — those will
trivially correlate. Drop them to surface the **real** behavioral signal.

Dropped: `avg_tip_count, avg_box_count, avg_wheel_count, avg_tip_usd, avg_box_usd, avg_wheel_usd` + `breadth_score`.

In [ ]:
tainted = ['avg_tip_count', 'avg_box_count', 'avg_wheel_count', 'avg_tip_usd', 'avg_box_usd', 'avg_wheel_usd', 'breadth_score']
clean_feats = [f for f in numeric_features if f not in tainted and f != 'avg_gift_usd']
print('Clean features :', clean_feats)

cols = clean_feats + ['avg_gift_usd']
corr = pool_u[cols].corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, annot=False)
plt.title(f'Spearman (clean) — Donate pool (n={len(pool_u):,})')
plt.tight_layout(); plt.show()

signed = corr['avg_gift_usd'].drop('avg_gift_usd')
top = signed.reindex(signed.abs().sort_values(ascending=False).index).head(15)
print('\nTop corr with avg_gift_usd (clean, signed, sorted by |corr|):')
print(top.round(3))


### 4c. Clean correlation — top-tail whales removed

Spearman is rank-based, so a few mega-gifters can't mathematically distort it — but the **top 1% of `avg_gift_usd`** can still represent a behaviorally different sub-population (one-shot whales vs steady spenders). Re-run the clean correlation after dropping users above the 99th percentile of `avg_gift_usd` to see how much of the signal comes from the long tail.


In [ ]:
# Trim outliers: drop users above the 99th percentile of avg_gift_usd
cap = pool_u['avg_gift_usd'].quantile(0.99)
trimmed = pool_u[pool_u['avg_gift_usd'] <= cap].copy()
print(f'99th-pct cap on avg_gift_usd : ${cap:,.2f}')
print(f'Users kept                   : {len(trimmed):,} / {len(pool_u):,} '
      f'({len(trimmed)/len(pool_u):.1%})')

cols = clean_feats + ['avg_gift_usd']
corr_t = trimmed[cols].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr_t, cmap='coolwarm', center=0, vmin=-1, vmax=1, annot=False)
plt.title(f'Spearman (clean, no top-1%) — Donate pool (n={len(trimmed):,})')
plt.tight_layout(); plt.show()

signed_t = corr_t['avg_gift_usd'].drop('avg_gift_usd')
top_t = signed_t.reindex(signed_t.abs().sort_values(ascending=False).index).head(15)

# Side-by-side vs the full-pool clean correlation
signed_full = pool_u[cols].corr(method='spearman')['avg_gift_usd'].drop('avg_gift_usd')
compare = pd.DataFrame({'full': signed_full, 'no_top_1pct': signed_t})
compare['delta'] = compare['no_top_1pct'] - compare['full']
compare = compare.reindex(compare['no_top_1pct'].abs().sort_values(ascending=False).index)
print('\nClean correlation with avg_gift_usd — full vs trimmed:')
print(compare.round(3))


## 5. Stickiness — how persistently users stay in the Donate pool

- `months_observed` — total active months in the 6-month window
- Repeat rate — share of pool users observed in ≥ N months
- Retention curve — monthly cohort follow-through
- Stickiness × intensity — do high-intensity users also stick around longer?

In [ ]:
# Distribution of months_observed
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
mo = pool_u['months_observed'].value_counts().sort_index()
mo_pct = mo / mo.sum()

sns.barplot(x=mo.index, y=mo.values, ax=axes[0], color='#4c78a8')
axes[0].set_title('Users by months_observed — Donate pool')
axes[0].set_xlabel('months_observed'); axes[0].set_ylabel('users')

cum = mo_pct.sort_index(ascending=False).cumsum().sort_index()
sns.lineplot(x=cum.index, y=cum.values, marker='o', ax=axes[1], linewidth=2)
axes[1].set_title('Share active in ≥ N months')
axes[1].set_xlabel('N months'); axes[1].set_ylabel('share of pool')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.tight_layout(); plt.show()

print('Repeat-rate table:')
print(pd.DataFrame({'months_observed': mo.index,
                    'users': mo.values,
                    'share': mo_pct.values.round(3)}).to_string(index=False))


In [ ]:
# Stickiness × intensity — do Q4 users stay longer?
stick = (pool_u.groupby('intensity')['months_observed']
               .agg(['mean', 'median', 'count']).round(2)
               .loc[intensity_order])
print('months_observed by intensity quartile:')
print(stick)

plt.figure(figsize=(7, 4))
sns.boxplot(data=pool_u, x='intensity', y='months_observed',
            order=intensity_order, showfliers=False)
plt.title('Donate pool — months_observed by intensity quartile')
plt.tight_layout(); plt.show()


In [ ]:
# Retention curve — for each monthly cohort (first month a user appears
# in the pool within the 6-month window), share still active in subsequent months.
mp = pool_m[['cust_id', 'data_month']].drop_duplicates().copy()
mp['data_month'] = pd.to_datetime(mp['data_month'])
first_seen = mp.groupby('cust_id')['data_month'].min().rename('cohort').reset_index()
mp = mp.merge(first_seen, on='cust_id')

months_sorted = sorted(mp['data_month'].unique())
month_idx = {m: i for i, m in enumerate(months_sorted)}
mp['period'] = mp['data_month'].map(month_idx) - mp['cohort'].map(month_idx)

cohort_size = mp.groupby('cohort')['cust_id'].nunique().rename('cohort_size')
ret = (mp.groupby(['cohort', 'period'])['cust_id'].nunique()
         .rename('active').reset_index()
         .merge(cohort_size, on='cohort'))
ret['retention'] = ret['active'] / ret['cohort_size']

pivot = ret.pivot(index='cohort', columns='period', values='retention').round(3)
print('Retention table (rows = first month in pool, cols = months since):')
print(pivot)

plt.figure(figsize=(9, 4))
for c in pivot.index:
    plt.plot(pivot.columns, pivot.loc[c].values, marker='o',
             label=pd.Timestamp(c).strftime('%Y-%m'))
plt.title('Donate pool — retention curve by cohort')
plt.xlabel('months since first appearance'); plt.ylabel('retention')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.legend(title='cohort', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()


## Takeaways scratch-pad — Donate pool

Fill in after running:
- Donate pool size: avg ≈ _ ; share of active users ≈ _
- Skew on `account_age_tier` / `watch_bucket` / `device_pref`: _
- Q4 (high-intensity) over-indexes on which segments? _
- Behavioral gap Q4 vs Q1 — biggest deltas: _
- Top features correlated with `avg_gift_usd` (clean): _
- Stickiness: median `months_observed` ≈ _ ; repeat-≥3 share ≈ _
- Stickiness × intensity: Q4 stays _ months on avg vs Q1 _ months
- Retention curve shape (sticky / leaky): _
